# Data Dictionary — Rename & Drop Columns

This notebook standardizes data freeze submission files by stripping each CSV down to the columns required by the submission guide, renaming them to a consistent schema, and exporting clean copies ready for submission.

---

### What it does
1. Loads all CSVs from the `dd_downloads/` folder (or a single file in standalone mode)
2. Strips each file to the core columns needed for submission
3. Renames columns to match the submission schema and adds a blank `units` column
4. Exports transformed files to the `output/` folder with clean, short filenames

---

### Required folder structure
For the notebook to run without any changes, your project should be organized like this:

```
datafreeze_ETL_scripts/
├── dd_downloads/               ← place downloaded CSV files here
├── output/                     ← transformed files saved here (auto-created)
└── python/
    └── data_dictionary.ipynb   ← you are here
```

> **Note:** CSV filenames in `dd_downloads/` can have any date suffix appended — the notebook finds them automatically and strips the date from the exported filename.

---

### Two ways to run

| Mode | When to use |
|---|---|
| `"folder"` | Process every CSV in `dd_downloads/` at once |
| `"standalone"` | Process a single file — useful for testing or a one-off submission if only one file was updated. |

Set your preferred mode in the **Configuration** cell below.

In [1]:
# Standard libraries used throughout the notebook.
# re handles the filename cleanup (stripping date codes).
# pathlib makes file path handling cleaner across operating systems.
import re
from pathlib import Path

import pandas as pd

## Configuration

Set your run mode and paths here before running the notebook.

In [2]:
# ── Run mode ──────────────────────────────────────────────────────────────────
# Set MODE to "folder" to process every CSV in dd_downloads/ at once.
# Set MODE to "standalone" to process a single file — update SINGLE_FILE with the path.
MODE        = "folder"
SINGLE_FILE = "../dd_downloads/uds-v4-ivp-ded-04142026.csv"  # only used when MODE = "standalone"

# ── Paths ─────────────────────────────────────────────────────────────────────
# These assume the notebook is running from inside the python/ folder.
# Adjust if you move the notebook elsewhere.
DOWNLOADS_DIR = Path("../dd_downloads")
OUTPUT_DIR    = Path("../output")

## Load CSVs

In [3]:
# In folder mode, every CSV in dd_downloads/ is picked up automatically —
# no need to rename or track files manually when a new date-stamped download arrives.
# In standalone mode, only the file you specified above is loaded.

if MODE == "standalone":
    path = Path(SINGLE_FILE)
    dataframes = {path.name: pd.read_csv(path)}
else:
    dataframes = {
        f.name: pd.read_csv(f)
        for f in sorted(DOWNLOADS_DIR.glob("*.csv"))
    }

# Print a quick summary so you can confirm the right files were picked up.
for name, df in dataframes.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

bds-v1-ded-10142025.csv: 52 rows, 8 columns
cls-v3-ded-10142025.csv: 19 rows, 8 columns
ftld-v3-fvp-ded-04142026.csv: 367 rows, 10 columns
ftld-v3-ivp-ded-04142026.csv: 367 rows, 10 columns
lbd-v3.0-fvp-ded-02252025.csv: 464 rows, 10 columns
lbd-v3.0-ivp-ded-02252025.csv: 463 rows, 10 columns
lbd-v3.1-fvp-ded-02252025.csv: 301 rows, 10 columns
lbd-v3.1-ivp-ded-02252025.csv: 300 rows, 10 columns
milestones-v3-ded-10142025 (1).csv: 33 rows, 8 columns
np-v11-ded-10142025 (1).csv: 164 rows, 8 columns
uds-v4-fvp-ded-04142026.csv: 1461 rows, 12 columns
uds-v4-ivp-ded-04142026.csv: 1535 rows, 10 columns


## Strip to core columns

In [4]:
# These are the only columns we need for the submission.
# Everything else (branching logic, missingness, notes, etc.) gets dropped here.
CORE_COLUMNS = ["packet", "var_name", "question", "conformity", "response_labels", "data_type"]

def strip_to_core(df):
    # Most forms have a "packet" column, but a small number use "form_name" instead.
    # When "packet" is missing we rename form_name to packet so the rest of the
    # pipeline stays consistent regardless of which forms are being processed.
    if "packet" not in df.columns:
        df = df.rename(columns={"form_name": "packet"})
        return df[CORE_COLUMNS], True
    return df[CORE_COLUMNS], False

defaulted = []
result = {}
for name, df in dataframes.items():
    result[name], used_form_name = strip_to_core(df)
    if used_form_name:
        defaulted.append(name)

dataframes = result

# This tells you which forms fell back to form_name instead of packet —
# useful to double-check nothing unexpected changed between data freeze versions.
print("Defaulted to form_name:", defaulted if defaulted else "none")

# Preview the first file to confirm the columns look right.
next(iter(dataframes.values())).head()

Defaulted to form_name: ['bds-v1-ded-10142025.csv', 'cls-v3-ded-10142025.csv', 'milestones-v3-ded-10142025 (1).csv', 'np-v11-ded-10142025 (1).csv']


,packet,var_name,question,conformity,response_labels,data_type
0,bds,adcid,0a. ADRC ID,List of current ADCIDs,List of current ADCIDs,Integer
1,bds,ptid,0b. PTID,String with max length of 10 characters,NaN,String
2,bds,visitdate,0c. Form Date,mm/dd/yyyy or yyyy/mm/dd,NaN,Date
3,bds,initials,0d. Examiner's initials,Any text,NaN,String
4,bds,formver,0e. Form version number,1,"1, 1",Numeric


## Rename columns & add Units

In [5]:
# Column names in the source files don't match the submission guide exactly.
# This mapping translates them, and we insert a blank "units" column in the
# right position since the guide requires it but the source data doesn't have one.
RENAME_MAP = {
    "var_name":   "variable_name",
    "question":   "description",
    "conformity": "key_description",
}

def rename_columns(df):
    df = df.rename(columns=RENAME_MAP)
    # Insert "units" as an empty column just before "key_description" to match
    # the order expected by the submission guide.
    df.insert(df.columns.get_loc("key_description"), "units", "")
    return df

dataframes = {name: rename_columns(df) for name, df in dataframes.items()}

# Preview to confirm the final column order before exporting.
next(iter(dataframes.values())).head()

,packet,variable_name,description,units,key_description,response_labels,data_type
0,bds,adcid,0a. ADRC ID,,List of current ADCIDs,List of current ADCIDs,Integer
1,bds,ptid,0b. PTID,,String with max length of 10 characters,NaN,String
2,bds,visitdate,0c. Form Date,,mm/dd/yyyy or yyyy/mm/dd,NaN,Date
3,bds,initials,0d. Examiner's initials,,Any text,NaN,String
4,bds,formver,0e. Form version number,,1,"1, 1",Numeric


## Export transformed data dictionaries

In [6]:
# Create the output folder if it doesn't exist yet — safe to run multiple times.
OUTPUT_DIR.mkdir(exist_ok=True)

def clean_filename(name):
    # Downloaded files have a date code baked into the name, e.g. 'uds-v4-ivp-ded-04142026.csv'.
    # We strip that out so the exported files have stable, readable names
    # that don't change every time a new freeze is downloaded.
    stem = Path(name).stem
    stem = re.sub(r"\s*\(\d+\)", "", stem)  # remove " (1)" artifacts from duplicate downloads
    stem = re.sub(r"-ded-\d+", "", stem)    # remove "-ded-MMDDYYYY" date suffix
    return stem + ".csv"

for name, df in dataframes.items():
    out_name = clean_filename(name)
    df.to_csv(OUTPUT_DIR / out_name, index=False)
    print(f"{name}  ->  {out_name}")

bds-v1-ded-10142025.csv  ->  bds-v1.csv
cls-v3-ded-10142025.csv  ->  cls-v3.csv
ftld-v3-fvp-ded-04142026.csv  ->  ftld-v3-fvp.csv
ftld-v3-ivp-ded-04142026.csv  ->  ftld-v3-ivp.csv
lbd-v3.0-fvp-ded-02252025.csv  ->  lbd-v3.0-fvp.csv
lbd-v3.0-ivp-ded-02252025.csv  ->  lbd-v3.0-ivp.csv
lbd-v3.1-fvp-ded-02252025.csv  ->  lbd-v3.1-fvp.csv
lbd-v3.1-ivp-ded-02252025.csv  ->  lbd-v3.1-ivp.csv
milestones-v3-ded-10142025 (1).csv  ->  milestones-v3.csv
np-v11-ded-10142025 (1).csv  ->  np-v11.csv
uds-v4-fvp-ded-04142026.csv  ->  uds-v4-fvp.csv
uds-v4-ivp-ded-04142026.csv  ->  uds-v4-ivp.csv
